[![Colab Badge](https://img.shields.io/badge/Open_in_Colab-blue?style=for-the-badge)][colab-link] [![Download Badge](https://img.shields.io/badge/Download-grey?style=for-the-badge)][download-link] [![JupyterHub](https://img.shields.io/badge/Jupyter_Hub-orange?style=for-the-badge)][jupyter-link]

[download-link]: https://github.com/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-05-01/1_hycom_generate_multifile_refs.ipynb
[colab-link]: https://colab.research.google.com/github/nmfs-opensci/nmfshackdays-2026/blob/main/topics/2026-05-01/1_hycom_generate_multifile_refs.ipynb
[jupyter-link]: https://nmfs-openscapes.2i2c.cloud/hub/user-redirect/lab?fromURL=https://raw.githubusercontent.com/nmfs-opensci/nmfshackdays-2026/main/topics/2026-05-01/1_hycom_generate_multifile_refs.ipynb


HYCOM data on AWS Open Data are stored in 63,341 NetCDF 64-bit offset files. The data in these files are stored as short integers with scale_factor and add_offset, but because these are not NetCDF4 files, there is no compression and no chunking. Each file contains one time step of data. 

We generate references for each file, and use kerchunk.utils.subchunk to create virtual chunks so that each vertical layer is treated as a chunk.  For this uncompressed data, the byte-ranges are the same for each file, so we only need to create references for one file and then clone that for all the files, changing only the URL and the time value. 

---

## Overall workflow

This notebook demonstrates a **kerchunk workflow** for converting many NetCDF files into a
**virtual Zarr dataset** without copying the data. The goal is to make large collections of
model output (like HYCOM) behave like a single cloud‑optimized dataset.

Key idea: **Kerchunk builds references, not data**

- The process reads NetCDF metadata.
- It records where each variable and chunk lives inside the file.
- The result is a mapping (JSON or Parquet) **kerchunk → byte range in original NetCDF file**

This means:

- No data duplication
- Very fast dataset assembly
- Works well with cloud storage

It enables:

- cloud‑native access
- fast subsetting
- scalable analysis


The mental model:

    NetCDF files
          ↓
    kerchunk reference generation
          ↓
    combined reference dataset
          ↓
    virtual Zarr dataset
          ↓
    xarray.open_dataset()


### Step 1. Build a template reference dictionary used across files

The files need to be structurally identical. For example:

    temperature(time, depth, lat, lon)

Each file needs to contain:

- the same variables
- the same grid
- the same dimensions
- a different time slice

The template dictionary defines the dataset *schema*. We will update the data file urls (to the netcdfs) in that dictionary.

### Step 2. Create a list of dictionaries

From the template, update the data file urls (to the netcdfs) in that dictionary to create a dictionary for each file.

### Step 3. Save the data dictionary list

Instead of saving one large JSON reference file (which is an option), this workflow stores references in Parquet.

- faster reads
- better scalability
- efficient metadata access
- works well with distributed systems

### Do all NetCDF files need to have the same structure to create a virtual Zarr dataset?

In most cases, yes — the NetCDF files used to build a virtual Zarr dataset (for example with kerchunk, VirtualiZarr, or MultiZarrToZarr) must have a consistent structure. 
The most important requirements are:

**1) Variables must be consistent**
The same variables should exist in each file (e.g., temperature, salinity). If a variable appears in some files but not others, the virtual dataset may fail to build or behave unpredictably.

**2) Dimensions must be compatible**
Dimensions like `lat`, `lon`, and `depth` usually need to be identical across files. The dimension along which files are combined — typically `time` — can differ in length, because that is the axis being concatenated.

**3) Coordinate systems must match**

- Same latitude and longitude values
- Same vertical levels
- Same projection or reference system

**4) Chunking and compression can differ**
These are storage details and usually do not need to match. Kerchunk reads the metadata and builds references regardless of how the data are chunked internally.

## Let's get started!

In [1]:
from kerchunk.netCDF3 import NetCDF3ToZarr
from kerchunk.combine import MultiZarrToZarr, auto_dask, JustLoad
from kerchunk.utils import subchunk, inline_array
from fsspec.implementations.reference import LazyReferenceMapper
import fsspec
import xarray as xr
import datetime as dt
import copy
import kerchunk
import base64
import struct
import numpy as np

In [2]:
%%time
fs = fsspec.filesystem('s3', anon=True)
flist = fs.glob('hycom-gofs-3pt1-reanalysis/*/*.nc')

CPU times: user 6.68 s, sys: 165 ms, total: 6.84 s
Wall time: 11.2 s


In [3]:
len(flist), flist[0]

(63341,
 'hycom-gofs-3pt1-reanalysis/1994/hycom_GLBv0.08_530_1994010112_t000.nc')

### Generate the dictionary for the file structure

We only need to do for one file since the byte layout is the same.

In [3]:
%%time
d0 = NetCDF3ToZarr("s3://" + flist[0], storage_options={"anon": True},
                  inline_threshold=400, version=2).translate()
# Subchunk the 4D data vars
for v in ['salinity', 'water_temp', 'water_u', 'water_v']:
    d0 = subchunk(store=d0, variable=v, factor=40)

CPU times: user 375 ms, sys: 62 ms, total: 437 ms
Wall time: 2.17 s


### Create a template dictionary from the first file in the dataset

In [4]:
# Storage options (for accessing the NetCDF files from AWS)
so = dict(anon=True, skip_instance_cache=True)
ds = xr.open_dataset(d0, engine='kerchunk', chunks={}, drop_variables='tau', 
                     backend_kwargs=dict(storage_options=dict(
                    remote_protocol='s3', lazy=False, remote_options=so)))

In [6]:
ds

<xarray.Dataset> Size: 19GB
Dimensions:            (time: 1, depth: 40, lat: 3251, lon: 4500)
Coordinates:
  * time               (time) datetime64[ns] 8B 1994-01-01T12:00:00
  * depth              (depth) float64 320B 0.0 2.0 4.0 ... 3e+03 4e+03 5e+03
  * lat                (lat) float64 26kB -80.0 -79.96 -79.92 ... 89.96 90.0
  * lon                (lon) float64 36kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    water_u            (time, depth, lat, lon) float64 5GB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_u_bottom     (time, lat, lon) float64 117MB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    water_v            (time, depth, lat, lon) float64 5GB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_v_bottom     (time, lat, lon) float64 117MB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    water_temp         (time, depth, lat, lon) float64 5GB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_temp_bottom  (time, lat, lon) float64 117MB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    salinity           (time, depth, lat, lon) float64 5GB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    salinity_bottom    (time, lat, lon) float64 117MB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    surf_el            (time, lat, lon) float64 117MB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
Attributes:
    classification_level:      UNCLASSIFIED
    distribution_statement:    Approved for public release. Distribution unli...
    downgrade_date:            not applicable
    classification_authority:  not applicable
    institution:               Naval Oceanographic Office
    source:                    HYCOM archive file
    history:                   archv2ncdf3z
    field_type:                instantaneous
    Conventions:               CF-1.6 NAVO_netcdf_v1.1

## Create functions to write dicts for all files from the template

We could repeat the code above, but 3 seconds x 63341 files is over 40 hours.

We need to define some functions to replace all the URLs in the reference dict `d0`.

In [51]:
d0["salinity/0.2.0.0"]

['s3://hycom-gofs-3pt1-reanalysis/1994/hycom_GLBv0.08_530_1994010112_t000.nc',
 3657441984,
 29259000]

In [4]:
def float_to_base64(number):
    # Pack the float into bytes
    packed = struct.pack('>d', number)
    
    # Encode the bytes to base64
    encoded = base64.b64encode(packed)
    # Convert bytes to string and return
    return encoded.decode('utf-8')

# Example usage
float_num = 122748.
encoded_str = float_to_base64(float_num)
print(f"Original number: {float_num}")
print(f"Base64 encoded: {encoded_str}")

Original number: 122748.0
Base64 encoded: QP33wAAAAAA=


In [5]:
def replace_first_item(d, target_string, replacement_string):
    for key, value in d.items():
        if isinstance(value, dict):
            # Recursively process nested dictionaries
            replace_first_item(value, target_string, replacement_string)
        elif isinstance(value, list) and value and isinstance(value[0], str):
            # Check if the value is a non-empty list and the first item is a string
            if value[0] == target_string:
                value[0] = replacement_string
    return d
#replace_first_item(d, f's3://{flist[0]}', f's3://{flist[1]}')

Function to generate the time from the filename

In [6]:
def name2date(f):
    year = f[51:55]
    month = f[55:57]
    day = f[57:59]
#    hour = f[59:61]  #always 12 for this dataset
    tau = f[64:66]
    return dt.datetime(int(year), int(month), int(day), int(tau))

Loop through all the files, generating the references for each file by replacing the URL and date in the reference dict template

In [10]:
%%time
dlist = []
time0 = dt.datetime(2000,1,1,0)
for i,v in enumerate(flist):
    dmod = copy.deepcopy(d0)
    time1 = name2date(v) + dt.timedelta(hours=12)
    time_val = (time1 - time0).total_seconds()/3600 
    encoded_str = float_to_base64(time_val)
    dmod['time/0'] = f'base64:{encoded_str}'
    dmod = replace_first_item(dmod, f's3://{flist[0]}',f's3://{v}')
    dlist.append(dmod)

CPU times: user 32.4 s, sys: 749 ms, total: 33.1 s
Wall time: 33.1 s


In [7]:
%%time
import json
import struct
import base64

dlist = []
time0 = dt.datetime(2000,1,1,0)

for i, v in enumerate(flist):
    dmod = copy.deepcopy(d0)

    time1 = name2date(v) + dt.timedelta(hours=12)
    time_val = (time1 - time0).total_seconds()/3600 

    encoded_str = float_to_base64(time_val)

    # --- ADD THESE 3 LINES ---
    z = json.loads(dmod["time/.zarray"])
    z["compressor"] = None
    dmod["time/.zarray"] = json.dumps(z)
    # -------------------------

    dmod["time/0"] = f"base64:{encoded_str}"

    dmod = replace_first_item(
        dmod,
        f"s3://{flist[0]}",
        f"s3://{v}"
    )

    dlist.append(dmod)

CPU times: user 30 s, sys: 748 ms, total: 30.7 s
Wall time: 30.8 s


In [12]:
len(dlist)

63341

In [8]:
import nest_asyncio
nest_asyncio.apply()

In [9]:
combined_parquet = 'hycom.parq'

In [10]:
out2 = LazyReferenceMapper.create(combined_parquet, fs=None, record_size=100000)

In [11]:
%%time
_ = MultiZarrToZarr(
        dlist,
        remote_protocol="s3",
        concat_dims="time",
        identical_dims=['lon', 'lat', 'depth'],
        preprocess=kerchunk.combine.drop("tau"),
        out=out2).translate()
out2.flush()

CPU times: user 1min 59s, sys: 7.77 s, total: 2min 7s
Wall time: 2min 5s


KeyError: 'lat/.zarray'

([], 0)

In [17]:
refs_test = MultiZarrToZarr(
    dlist[:3],
    remote_protocol="s3",
    concat_dims=["time"],
    identical_dims=["lon", "lat", "depth"],
    preprocess=kerchunk.combine.drop("tau"),
).translate()

In [ ]:
# Rich wrote to a s3 bucket
# fs_write = fsspec.filesystem('s3', profile='osn-esip', skip_instance_cache=True, use_listings_cache=False,
#                             client_kwargs={'endpoint_url': 'https://ncsa.osn.xsede.org'})
# combined_parquet_aws = 's3://esip/rsignell/hycom.parq'
# fs_write.rm(combined_parquet_aws, recursive=True)    # delete any existing refs on OSN
# _ = fs_write.upload(combined_parquet, combined_parquet_aws, recursive=True)  # upload refs to OSN
# Check to make sure the references got updated
# fs_write.info(f'{combined_parquet_aws}/.zmetadata')

#### Open the references for the entire dataset

Target options (for accessing the reference files from OSN)

In [18]:
combined_parquet_aws = 's3://esip/rsignell/hycom.parq'
so = dict(anon=True, skip_instance_cache=True)
to = dict(anon=True, skip_instance_cache=True, 
          client_kwargs={'endpoint_url': 'https://ncsa.osn.xsede.org'})

In [19]:
ds = xr.open_dataset(combined_parquet_aws, engine='kerchunk', chunks={},
                    backend_kwargs=dict(storage_options=dict(target_options=to,
                    remote_protocol='s3', lazy=True, remote_options=so)))

In [20]:
ds

<xarray.Dataset> Size: 1PB
Dimensions:            (time: 63341, depth: 40, lat: 3251, lon: 4500)
Coordinates:
  * time               (time) datetime64[ns] 507kB 1994-01-01T12:00:00 ... 20...
  * depth              (depth) float64 320B 0.0 2.0 4.0 ... 3e+03 4e+03 5e+03
  * lat                (lat) float64 26kB -80.0 -79.96 -79.92 ... 89.96 90.0
  * lon                (lon) float64 36kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    salinity           (time, depth, lat, lon) float64 297TB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    salinity_bottom    (time, lat, lon) float64 7TB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    surf_el            (time, lat, lon) float64 7TB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    water_temp         (time, depth, lat, lon) float64 297TB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_temp_bottom  (time, lat, lon) float64 7TB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    water_u            (time, depth, lat, lon) float64 297TB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_u_bottom     (time, lat, lon) float64 7TB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
    water_v            (time, depth, lat, lon) float64 297TB dask.array<chunksize=(1, 1, 3251, 4500), meta=np.ndarray>
    water_v_bottom     (time, lat, lon) float64 7TB dask.array<chunksize=(1, 3251, 4500), meta=np.ndarray>
Attributes:
    Conventions:               CF-1.6 NAVO_netcdf_v1.1
    classification_authority:  not applicable
    classification_level:      UNCLASSIFIED
    distribution_statement:    Approved for public release. Distribution unli...
    downgrade_date:            not applicable
    field_type:                instantaneous
    history:                   archv2ncdf3z
    institution:               Naval Oceanographic Office
    source:                    HYCOM archive file

## Make a plot

In [1]:
%%time
da = ds['water_temp'].sel(depth=0, time='2012-10-29 17:00', method='nearest').load()

CPU times: user 7 μs, sys: 0 ns, total: 7 μs
Wall time: 9.3 μs


NameError: name 'ds' is not defined

In [ ]:
da.hvplot.quadmesh(x='lon', y='lat', geo=True, global_extent=True, tiles='ESRI', cmap='viridis', rasterize=True)